In [2]:
import pandas as pd
df = pd.read_csv("paysim_10k_sample.csv")
print(df.shape)
df.head()

(10000, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,180,CASH_OUT,114664.45,C1911904419,0.00,0.00,C763019604,163667.97,278332.42,0,0
1,35,CASH_OUT,15740.84,C2009416105,0.00,0.00,C1391458307,772346.04,788086.88,0,0
2,403,PAYMENT,19355.33,C971312477,12231.00,0.00,M1617207043,0.00,0.00,0,0
3,18,CASH_OUT,160747.90,C1324033502,842914.33,682166.43,C849757077,1038016.75,859475.31,0,0
4,35,PAYMENT,22804.93,C1774451797,0.00,0.00,M766297664,0.00,0.00,0,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   step            10000 non-null  int64  
 1   type            10000 non-null  object 
 2   amount          10000 non-null  float64
 3   nameOrig        10000 non-null  object 
 4   oldbalanceOrg   10000 non-null  float64
 5   newbalanceOrig  10000 non-null  float64
 6   nameDest        10000 non-null  object 
 7   oldbalanceDest  10000 non-null  float64
 8   newbalanceDest  10000 non-null  float64
 9   isFraud         10000 non-null  int64  
 10  isFlaggedFraud  10000 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 859.5+ KB


In [4]:
df.isnull().sum()

,0
step,0
type,0
amount,0
nameOrig,0
oldbalanceOrg,0
newbalanceOrig,0
nameDest,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,10000.000000,1.000000e+04,1.000000e+04,1.000000e+04,1.000000e+04,1.000000e+04,10000.000000,10000.0000
mean,250.287600,2.424117e+05,8.745570e+05,8.265502e+05,1.076717e+06,1.226878e+06,0.050000,0.0001
std,149.358776,7.961442e+05,2.929251e+06,2.890649e+06,3.517334e+06,3.743513e+06,0.217956,0.0100
min,1.000000,6.000000e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0000
25%,155.000000,1.470649e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0000
50%,250.000000,8.518716e+04,1.818200e+04,0.000000e+00,1.220957e+05,2.140559e+05,0.000000,0.0000
75%,348.000000,2.267622e+05,1.469898e+05,1.292462e+05,9.022742e+05,1.102295e+06,0.000000,0.0000
max,741.000000,3.147911e+07,3.567455e+07,3.285196e+07,1.667418e+08,1.680585e+08,1.000000,1.0000


In [9]:
print(df["isFraud"].value_counts())
print(df.groupby("type")["isFraud"].sum())

isFraud
0    9500
1     500
Name: count, dtype: int64
type
CASH_IN       0
CASH_OUT    254
DEBIT         0
PAYMENT       0
TRANSFER    246
Name: isFraud, dtype: int64


In [10]:
df["hour_of_day"] = df["step"] % 24
df["day"] = df["step"] // 24 + 1

In [11]:
df["orig_emptied"] = ((df["oldbalanceOrg"] > 0) & (df["newbalanceOrig"] == 0)).astype(int)

In [12]:
df["balance_error"] = (df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]).round(2)
df["amount_to_balance"] = (df["amount"] / df["oldbalanceOrg"]).where(df["oldbalanceOrg"] > 0).round(3)

In [13]:
high_amt = df["amount"].quantile(0.90)
print(high_amt)

df["high_amount"] = (df["amount"] > high_amt).astype(int)
df["risky_type"] = df["type"].isin(["TRANSFER", "CASH_OUT"]).astype(int)

410914.205


In [14]:
df["risk_score"] = df["risky_type"] + df["orig_emptied"] * 2 + df["high_amount"]

def risk_level(score):
    if score >= 3:
        return "High"
    elif score == 2:
        return "Medium"
    return "Low"

df["risk_level"] = df["risk_score"].apply(risk_level)

In [15]:
df["risk_level"].value_counts()
df.groupby("risk_level")["isFraud"].agg(["count", "sum", "mean"]).round(3)

,count,sum,mean
risk_level,,,
High,2249,486,0.216
Low,6917,2,0.000
Medium,834,12,0.014


In [16]:
print(df.shape)
df.isnull().sum()

(10000, 20)


,0
step,0
type,0
amount,0
nameOrig,0
oldbalanceOrg,0
newbalanceOrig,0
nameDest,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0


In [17]:
df["amount_to_balance"] = df["amount_to_balance"].fillna(0)

In [18]:
df.to_csv("transactions_clean.csv", index=False)

In [19]:
from google.colab import files
files.download("transactions_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>